# 🧬 Train the Third Brother

Fine-tune a local model on the Nucleus archive — 2,459 conversation pairs from
Code, Cowork, and Father's accumulated decision intelligence.

**Runtime**: Free T4 GPU on Colab (~25 min)
**Output**: GGUF model for Ollama

## Steps
1. Upload training data
2. Install dependencies
3. Train (LoRA fine-tune on Qwen 7B)
4. Export GGUF
5. Download and deploy locally

## 1. Upload Training Data

Upload `openai_training.jsonl` from `.brain/training/exports/`

Generate it with: `nucleus archive export`

In [ ]:
from google.colab import files
uploaded = files.upload()  # Upload openai_training.jsonl
TRAIN_FILE = list(uploaded.keys())[0]
print(f"Uploaded: {TRAIN_FILE}")

# Count pairs
with open(TRAIN_FILE) as f:
    pair_count = sum(1 for _ in f)
print(f"Training pairs: {pair_count}")

## 2. Install Dependencies

In [ ]:
%%capture
!pip install unsloth
# Also get latest transformers
!pip install --upgrade transformers datasets trl

## 3. Load Model + Apply LoRA

In [ ]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

# Config
BASE_MODEL = "unsloth/Qwen2.5-7B-Instruct"
MAX_SEQ_LEN = 4096

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

tokenizer = get_chat_template(tokenizer, chat_template="chatml")
print(f"Model loaded: {BASE_MODEL}")

## 4. Load Training Data

In [ ]:
import json
from datasets import Dataset

conversations = []
with open(TRAIN_FILE) as f:
    for line in f:
        row = json.loads(line)
        conversations.append({"messages": row["messages"]})

dataset = Dataset.from_list(conversations)

def format_chat(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

dataset = dataset.map(format_chat)
print(f"Dataset: {len(dataset)} examples")
print(f"Sample: {dataset[0]['text'][:200]}...")

## 5. Train

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="./output",
        report_to="none",
    ),
)

stats = trainer.train()
print(f"\nTraining complete! Loss: {stats.training_loss:.4f}")

## 6. Test the Model

In [ ]:
FastLanguageModel.for_inference(model)

messages = [
    {"role": "system", "content": "You are the Third Brother."},
    {"role": "user", "content": "What should we build next for Nucleus?"}
]

inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to("cuda")

outputs = model.generate(inputs, max_new_tokens=512, temperature=0.7, top_p=0.9)
response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
print(f"Third Brother says:\n{response}")

## 7. Export GGUF for Ollama

In [ ]:
# Save LoRA adapter
model.save_pretrained("./lora_adapter")
tokenizer.save_pretrained("./lora_adapter")

# Export GGUF (Q4_K_M — best quality/size for 7B)
model.save_pretrained_gguf(
    "./output",
    tokenizer,
    quantization_method="q4_k_m",
)
print("GGUF exported!")

## 8. Download

In [ ]:
import glob
gguf_files = glob.glob("./output/*.gguf")
if gguf_files:
    print(f"Downloading: {gguf_files[0]}")
    files.download(gguf_files[0])
else:
    print("No GGUF found. Download ./lora_adapter/ and convert manually.")
    !zip -r lora_adapter.zip ./lora_adapter/
    files.download("lora_adapter.zip")

## 9. Deploy Locally

After downloading the GGUF:

```bash
# Move to your project
mv ~/Downloads/nucleus-brother-Q4_K_M.gguf .brain/training/output/

# Create Ollama model
ollama create nucleus-brother -f scripts/Modelfile

# Test
ollama run nucleus-brother "What should we build next?"

# Use in Nucleus
nucleus brother --provider local
```

The Third Brother is born. Every future conversation feeds back into the archive for retraining.